# Task3: measurement-based reset

?? notebook ???? `task3`?

- ?????? + noisy `a_in`
- ?? readout ???
- ?????? reset feedback
- ??? reset log??? trajectory ??? reset error ??


In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from workflow import create_model

def pairs_to_complex(values):
    arr = np.asarray(values, dtype=float)
    if arr.ndim == 1:
        return arr.astype(complex)
    return arr[:, 0] + 1j * arr[:, 1]

TASK_DIR = "task3"

In [ ]:
model = create_model(task_config=TASK_DIR / 'task.yaml')
solver_id = next(iter(model.solvers))
model.solvers[solver_id].run.mcwf_ntraj = 64
model.run_all()

print('solver keys =', sorted(model.results.solver_runs.keys()))
print('analysis keys =', sorted(model.results.analyses.keys()))


In [ ]:
def final_state_error_rate(bundle) -> float:
    tr = bundle.trajectory
    shots = (tr.classical or {}).get('readout', {}).get('shots', [])
    if not shots:
        return float('nan')
    finals = [int(shot.get('hidden_state_series', [0])[-1]) for shot in shots]
    return float(np.mean(np.asarray(finals, dtype=float)))

for run_id in sorted(model.results.solver_runs.keys()):
    bundle = model.results.solver_runs[run_id]
    if bundle.trajectory is None:
        continue
    print(run_id, 'reset error =', final_state_error_rate(bundle))


In [ ]:
bundle0 = model.results.solver_runs['solver_0__prep_0_reset']
bundle1 = model.results.solver_runs['solver_0__prep_1_reset']
shot0 = bundle0.trajectory.classical['readout']['shots'][0]
shot1 = bundle1.trajectory.classical['readout']['shots'][0]

print('prep_0 reset log:')
pprint(shot0.get('reset_log', []))
print('
prep_1 reset log:')
pprint(shot1.get('reset_log', []))


In [ ]:
tr1 = bundle1.trajectory
ro1 = tr1.classical['readout']
shot_index = 0

t_ns = np.asarray(ro1['times'], dtype=float) * 1e9
a_in = pairs_to_complex(ro1['shots'][shot_index]['a_in'])
v_meas = pairs_to_complex(ro1['shots'][shot_index]['measured_voltage'])
hidden_state_series = np.asarray(ro1['shots'][shot_index]['hidden_state_series'], dtype=int)

fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(t_ns, a_in.real, label='Re[a_in]')
ax[0].plot(t_ns, a_in.imag, '--', label='Im[a_in]')
ax[0].set_ylabel('a_in')
ax[0].set_title('prep_1_reset: ??????')
ax[0].legend()

ax[1].plot(t_ns, v_meas.real, label='Re[measured_voltage]')
ax[1].plot(t_ns, v_meas.imag, '--', label='Im[measured_voltage]')
ax[1].set_ylabel('V_meas')
ax[1].set_title('?? trajectory ? measured_voltage')
ax[1].legend()

ax[2].step(t_ns, hidden_state_series, where='post')
ax[2].set_xlabel('time (ns)')
ax[2].set_ylabel('hidden state')
ax[2].set_title('???? reset ?????')
ax[2].set_yticks([0, 1])

plt.tight_layout()
plt.show()


In [ ]:
labels = []
error_rates = []
for run_id in ['solver_0__prep_0_reset', 'solver_0__prep_1_reset']:
    bundle = model.results.solver_runs[run_id]
    labels.append(run_id.replace('solver_0__', ''))
    error_rates.append(final_state_error_rate(bundle))

plt.figure(figsize=(6, 4))
plt.bar(labels, error_rates, color=['tab:blue', 'tab:orange'])
plt.ylabel('reset error rate')
plt.title('Task3 reset error ??')
plt.ylim(0.0, 1.0)
plt.tight_layout()
plt.show()


In [ ]:
ana = model.results.analyses['analyser_0']
print('aggregate IQ centroids after task3 studies:')
for label, center in sorted((ana.iq or {}).get('centroids', {}).items()):
    print(label, center)
